In [ ]:
from datetime import datetime
from urllib.parse import quote_plus
import pandas as pd
from sqlalchemy import create_engine
import os

load_dotenv()  

DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_NAME = os.getenv("DB_NAME", "olist_dw")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}/{DB_NAME}")

# Tab 1: Executive Monthly KPI & Data Health Summary
df_exec = pd.read_sql("""
    SELECT 
        order_month AS Month,
        total_orders AS Total_Orders,
        leaked_orders AS Canceled_Unavailable_Orders,
        leaked_gmv AS Leaked_GMV_BRL,
        data_completeness_pct AS Data_Completeness_Pct
    FROM vw_data_health_metrics
    ORDER BY order_month DESC
""", engine)



# Tab 2: Ops Exception List - Top 25 Late-Delivering Sellers
df_late_sellers = pd.read_sql("""
    SELECT 
        oi.seller_id,
        COUNT(DISTINCT oi.order_id) AS total_delivered_orders,
        SUM(CASE WHEN o.sla_status = 'Late' THEN 1 ELSE 0 END) AS late_orders,
        ROUND(SUM(CASE WHEN o.sla_status = 'Late' THEN 1 ELSE 0 END) * 100.0 / COUNT(DISTINCT oi.order_id), 2) AS late_rate_pct,
        ROUND(AVG(o.review_score), 2) AS avg_review_score
    FROM vw_fact_order_items oi
    JOIN vw_fact_orders o ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY oi.seller_id
    HAVING COUNT(DISTINCT oi.order_id) >= 30
    ORDER BY late_rate_pct DESC
    LIMIT 25
""", engine)

# Tab 3: Growth Win-Back Target List - Top 100 At-Risk High Spenders (RFM)
df_rfm_winback = pd.read_sql("""
    SELECT 
        customer_unique_id,
        customer_state,
        recency_days,
        frequency,
        monetary AS lifetime_spend_brl,
        rfm_cell,
        customer_segment
    FROM vw_customer_rfm
    WHERE customer_segment = 'At-Risk High Spenders'
    ORDER BY monetary DESC
    LIMIT 100
""", engine)

# Tab 4: B2B Marketing Funnel Channel Summary
df_funnel_mis = pd.read_sql("""
    SELECT 
        acquisition_channel,
        COUNT(*) AS total_mqls,
        SUM(is_closed_won) AS closed_won_deals,
        ROUND(AVG(is_closed_won) * 100, 2) AS win_rate_pct,
        ROUND(AVG(days_to_close), 1) AS avg_days_to_close,
        ROUND(SUM(seller_delivered_gmv), 2) AS total_live_gmv_brl
    FROM vw_marketing_funnel
    GROUP BY acquisition_channel
    ORDER BY total_mqls DESC
""", engine)

# ---  Export & Format Styled Excel MIS Pack ---
output_file = f"Olist_Executive_MIS_Pack_{datetime.now().strftime('%Y%m%d')}.xlsx"

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    workbook = writer.book
    header_fmt = workbook.add_format({
        "bold": True, "bg_color": "#1E3A8A", "font_color": "#FFFFFF",
        "border": 1, "align": "center"
    })
    alert_fmt = workbook.add_format({"bg_color": "#FEE2E2", "font_color": "#991B1B"})

    sheets = {
        "1_Exec_Health_Summary": df_exec,
        "2_Ops_Late_Sellers_Alert": df_late_sellers,
        "3_CRM_Winback_Targets": df_rfm_winback,
        "4_B2B_Funnel_MIS": df_funnel_mis,
    }

    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1)
        ws = writer.sheets[sheet_name]
        ws.write(0, 0, f"Automated MIS Report | Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
                 workbook.add_format({"bold": True, "italic": True, "font_color": "#374151"}))

        for col_idx, col_name in enumerate(df.columns):
            ws.write(1, col_idx, col_name, header_fmt)
            max_len = max(df[col_name].astype(str).map(len).max(), len(col_name)) + 4
            ws.set_column(col_idx, col_idx, min(max_len, 35))

        # Highlight severe SLA breaches (>20% late rate) on Ops Exception tab
        if sheet_name == "2_Ops_Late_Sellers_Alert":
            ws.conditional_format(2, 4, len(df) + 1, 4, {
                "type": "cell", "criteria": ">=", "value": 20, "format": alert_fmt
            })

print(f"MIS Pack generated successfully: {output_file}")

MIS Pack generated successfully: Olist_Executive_MIS_Pack_20261123.xlsx
